# Best Predict Tutorial

This tutorial is for the ICAR Best Predict method for lactation milk yield. Using this tutorial and the example data the goal is to make the method understandable and easy to use. 

The tutorial walks through the Best Prediction workflow with a small dummy dataframe and shows the main options:

- the default `best_predict_method` run
- the single-lactation helper
- the step-by-step preprocessing logic
- options for custom column names
- fitting your own standard curve and covariance matrix from your own reference data
- passing your own curve and matrix directly

This notebook is based directly on the source module in `src/lactationcurve/characteristics/best_predict.py`.

Author: Meike van Leerdam

Date: 12-6-2026

In [1]:
# standard packages
import numpy as np
import pandas as pd

# import the functions to be tried out for best predict
from lactationcurve.characteristics.best_predict import (
    COV_MATRIX,
    STANDARD_CURVE,
    best_predict_method,
    best_predict_method_single_lac,
    fit_autocorrelation_matrix,
    fit_standard_lc,
    preprocess_measured_data,
)

## Create a dummy dataframe

The demo starts with a small panda dataframe that looks like real test-day milk recordings. We split it into a target lactation and a reference set so we can show both the default prediction path and the fit-from-reference options.

In [2]:
dummy_records = []
for test_id, base_yield in [
    (1001, 31.0),
    (1002, 29.5),
    (1003, 28.0),
    (1004, 26.8),
    (1005, 25.5),
]:
    for day_in_milk, adjustment in zip(
        [5, 20, 35, 55, 80, 110, 150, 190, 240, 290],
        [0.0, 2.1, 3.4, 2.6, 1.1, -0.4, -1.7, -2.5, -3.1, -3.6],
    ):
        dummy_records.append(
            {
                "TestId": test_id,
                "DaysInMilk": day_in_milk,
                "MilkingYield": round(base_yield + adjustment + (test_id - 1003) * 0.35, 1),
            }
        )

dummy_df = pd.DataFrame(dummy_records)

reference_df = dummy_df[dummy_df["TestId"].isin([1001, 1002, 1003, 1004])].copy()
target_df = dummy_df[dummy_df["TestId"] == 1005].copy()

dummy_df

,TestId,DaysInMilk,MilkingYield
0,1001,5,30.3
1,1001,20,32.4
2,1001,35,33.7
3,1001,55,32.9
4,1001,80,31.4
5,1001,110,29.9
6,1001,150,28.6
7,1001,190,27.8
8,1001,240,27.2
9,1001,290,26.7


## 1. Run Best Prediction with the package defaults

This is the shortest path through the code: `best_predict_method` standardizes the input, splits it by `TestId`, and predicts one cumulative 305-day yield per lactation using the standard curve and covariance matrix provided by the package. When you apply this to your own data set, you have to keep in mind that the a certain shape of the lactationcurve is assumed, so the method will work best if this ressembles the shape of the lactation curve of your own animals.

In [3]:
default_predictions = best_predict_method(dummy_df.copy())
default_predictions

,TestId,LactationMilkYield
0,1001,8953.177055
1,1002,8599.612083
2,1003,8264.795805
3,1004,8025.358849
4,1005,7726.062653


## 2. Predict one lactation directly

`best_predict_method_single_lac` is the lower-level entry point. It expects a single lactation dataframe and returns one cumulative 305-day prediction as a float.

In [4]:
single_prediction = best_predict_method_single_lac(target_df.copy())
print(f"The result of the single lactation curve prediction is: {single_prediction:.3e} kg")

The result of the single lactation curve prediction is: 7.726e+03 kg


## 3. Look at the preprocessing step

Internally the function subtracts the standard curve from the observed test-day milk yields and builds a 305-day deviation vector (Measured milk yield - expected milk yield). In the final method all these deviations will be added to the standard lactation curve and the expected deviations of the days that were not measured but predicted. Unobserved days are filled with zero before the covariance projection happens.

In [5]:
observed_lactation = target_df.sort_values("DaysInMilk").copy()
corrected_series = preprocess_measured_data(observed_lactation, STANDARD_CURVE)

print(observed_lactation)
print()
print("Deviation vector on the observed days:")
print(corrected_series.loc[observed_lactation["DaysInMilk"]].to_dict())
print()
print("Total deviation added to the standard curve:")
print(float(corrected_series.sum()))

    TestId  DaysInMilk  MilkingYield  MilkDifference
40    1005           5          26.2       -3.257007
41    1005          20          28.3       -9.617181
42    1005          35          29.6      -10.991868
43    1005          55          28.8      -12.831907
44    1005          80          27.3      -13.884659
45    1005         110          25.8      -13.682524
46    1005         150          24.5      -11.881648
47    1005         190          23.7       -9.277467
48    1005         240          23.1       -5.667042
49    1005         290          22.6       -2.247153

Deviation vector on the observed days:
{5: -3.2570072497460743, 20: -9.617181222726611, 35: -10.991868221546447, 55: -12.831907405470002, 80: -13.88465863803204, 110: -13.682524464605006, 150: -11.881648000604642, 190: -9.277467499199748, 240: -5.66704157325529, 290: -2.247152828105719}

Total deviation added to the standard curve:
-93.33845710329159


## 4. Use custom column names

The function can standardize different input schemas. Here the dataframe is renamed to `AnimalId`, `DIM`, and `Yield`, and the column mapping is passed explicitly. 

When using your own column name schema, be careful that all the TestIds are considered as a seperate lactation. It is possible to use an AnimalId as a TestId, however if you have multiple lactations in your dataset, the same AnimalId can have multiple lactations for different parities. So then it is better to make a new column TestIds with a unique identifier for each lactation. 

In [6]:
aliased_df = dummy_df.rename(
    columns={
        "TestId": "AnimalId",
        "DaysInMilk": "DIM",
        "MilkingYield": "Yield",
    }
)

aliased_predictions = best_predict_method(
    aliased_df.copy(),
    days_in_milk_col="DIM",
    milking_yield_col="Yield",
    test_id_col="AnimalId",
)

aliased_predictions

,TestId,LactationMilkYield
0,1001,8953.177055
1,1002,8599.612083
2,1003,8264.795805
3,1004,8025.358849
4,1005,7726.062653


## 5. Fit the curve and covariance matrix from reference data

This mirrors the more advanced branch in `best_predict.py`: fit a standard curve from your own  reference set, estimate the covariance structure from the same reference dataframe, and then reuse those fitted ingredients for the target lactation. The reference dataset can be a different dataset then of the lactations that you want to predict the lactation yield for, but it can also be the same. The reference dataset looks at aggregated lactationcurves. 

In [35]:
fitted_standard_curve = fit_standard_lc(reference_df.copy())
fitted_covariance = fit_autocorrelation_matrix(reference_df.copy(), fitted_standard_curve)["B_hat"]

print("The fitted matrix has the following parameter values:")
print("b1:", fit_autocorrelation_matrix(reference_df.copy(), fitted_standard_curve)["b1"])
print("b2:", fit_autocorrelation_matrix(reference_df.copy(), fitted_standard_curve)["b2"])
print("rho:", fit_autocorrelation_matrix(reference_df.copy(), fitted_standard_curve)["rho"])

fitted_reference_predictions = best_predict_method(
    target_df.copy(),
    standard_lc=fitted_standard_curve,
    covariance_matrix=fitted_covariance,
)

fitted_reference_predictions

The fitted matrix has the following parameter values:
b1: 1.332981071170676e-05
b2: 2.1303782778022606
rho: 0.9925947735916103


,TestId,LactationMilkYield
0,1005,7679.532621


## 6. Fit everything inside `best_predict_method`

If you already have a reference dataframe, you can let the dispatcher fit the covariance matrix for you by setting `fit_standard_lc_from_data=True` and passing `reference_df`.

In [8]:
auto_fit_predictions = best_predict_method(
    target_df.copy(),
    fit_standard_lc_from_data=True,
    reference_df=reference_df.copy(),
)

auto_fit_predictions

,TestId,LactationMilkYield
0,1005,7684.051893


## 7. Pass your own standard curve and covariance matrix directly

This is the most explicit option in the source file: use the same single-lactation helper, but swap in your own curve and matrix without fitting them inside the function.

In [9]:
custom_standard_curve = STANDARD_CURVE * 0.98
custom_covariance_matrix = COV_MATRIX.copy()
custom_covariance_matrix = (
    custom_covariance_matrix * 0.9 + np.eye(custom_covariance_matrix.shape[0]) * 0.1
)

custom_prediction = best_predict_method_single_lac(
    target_df.copy(),
    standard_lc=custom_standard_curve,
    covariance_matrix=custom_covariance_matrix,
)

custom_prediction

np.float64(7723.071781076493)

# 8. Visualize the standard matrix and standard curve used by the platform

The current implementation of best predict only uses one standard lactation curve with one corresponding matrix. Later we will add multiple curves and stratify lactations based on for instance parity, as not every cow has the same shape of the lactation curve. In your own data pipeline this is already possible if you stratify your reference data before putting it into the function, as then only these lactations will be taken into account to make the curve and matrix. 

In [13]:
print("The Wood standard lactation curve currently used in the best predict method is:")
print(STANDARD_CURVE)

The Wood standard lactation curve currently used in the best predict method is:
[20.88852108 24.28756033 26.48529607 28.13333695 29.45700725 30.56351807
 31.51304545 32.34304569 33.0785599  33.7372317  34.33200401 34.8726797
 35.36687887 35.82065388 36.23889995 36.62563841 36.98421725 37.31745679
 37.62775749 37.91718122 38.18751364 38.44031259 38.67694623 38.89862343
 39.10641824 39.3012897  39.4840982  39.6556189  39.81655295 39.9675369
 40.10915064 40.24192411 40.36634304 40.48285392 40.59186822 40.69376607
 40.78889947 40.87759509 40.96015672 41.03686741 41.1079914  41.17377578
 41.23445203 41.29023728 41.34133561 41.387939   41.4302284  41.46837449
 41.50253857 41.53287316 41.55952271 41.58262416 41.60230747 41.61869609
 41.63190741 41.64205315 41.64923978 41.65356877 41.65513699 41.65403694
 41.65035703 41.64418185 41.63559233 41.62466604 41.61147731 41.59609742
 41.57859483 41.55903524 41.53748181 41.51399526 41.48863401 41.46145429
 41.43251026 41.40185408 41.36953608 41.335604

In [14]:
print("The covariance matrix currently used in the best predict method is:")
print(COV_MATRIX)

The covariance matrix currently used in the best predict method is:
[[75.16786  67.28662  67.05099  ... 23.40674  23.32477  23.243088]
 [67.28662  75.16786  67.28662  ... 23.488997 23.40674  23.32477 ]
 [67.05099  67.28662  75.16786  ... 23.571543 23.488997 23.40674 ]
 ...
 [23.40674  23.488997 23.571543 ... 75.16786  67.28662  67.05099 ]
 [23.32477  23.40674  23.488997 ... 67.28662  75.16786  67.28662 ]
 [23.243088 23.32477  23.40674  ... 67.05099  67.28662  75.16786 ]]


## Wrap-up

Use `best_predict_method` for the normal multi-lactation workflow, `best_predict_method_single_lac` when you already have one lactation isolated, and the reference-fitting or direct-override branches when you want to control the standard curve and covariance matrix yourself.

[For further information please look at the package documentation on this topic.](https://bovi-analytics.github.io/bovi/lactationcurve/characteristics/best_predict.html)